# Self-Organizing Map for the Breast Cancer Dataset

This notebook trains a Self-Organizing Map (SOM) on the Wisconsin breast cancer dataset and saves an animation of the U-matrix as the SOM learns.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from minisom import MiniSom
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import MinMaxScaler

RANDOM_SEED = 42
MAP_SIZE = (12, 12)
N_ITERATIONS = 3000
N_FRAMES = 75
ANIMATION_PATH = Path("som_breast_cancer_animation.gif")

np.random.seed(RANDOM_SEED)

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target
labels = data.target_names[y]

feature_names = data.feature_names
df = pd.DataFrame(X, columns=feature_names)
df["diagnosis"] = labels

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

print(f"Samples: {X_scaled.shape[0]}")
print(f"Features: {X_scaled.shape[1]}")
print(pd.Series(labels).value_counts())
df.head()

In [ ]:
som = MiniSom(
    x=MAP_SIZE[0],
    y=MAP_SIZE[1],
    input_len=X_scaled.shape[1],
    sigma=1.5,
    learning_rate=0.5,
    neighborhood_function="gaussian",
    random_seed=RANDOM_SEED,
)

som.random_weights_init(X_scaled)

# MiniSom's train_random method is convenient, but a manual loop lets us
# capture U-matrix snapshots for the animation.
rng = np.random.default_rng(RANDOM_SEED)
training_order = rng.integers(0, X_scaled.shape[0], size=N_ITERATIONS)
snapshot_steps = set(np.linspace(0, N_ITERATIONS - 1, N_FRAMES, dtype=int))
snapshots = []

for step, sample_index in enumerate(training_order):
    sample = X_scaled[sample_index]
    winner = som.winner(sample)
    som.update(sample, winner, step, N_ITERATIONS)

    if step in snapshot_steps:
        snapshots.append((step + 1, som.distance_map().T.copy()))

print(f"SOM trained successfully with {N_ITERATIONS:,} iterations.")
print(f"Captured {len(snapshots)} animation frames.")

In [ ]:
plt.figure(figsize=(8, 8))
plt.pcolor(som.distance_map().T, cmap="viridis")
plt.colorbar(label="Mean neighbor distance")
plt.title("SOM U-Matrix - Breast Cancer Dataset")
plt.xlabel("SOM x")
plt.ylabel("SOM y")
plt.tight_layout()
plt.show()

In [ ]:
winners = np.array([som.winner(row) for row in X_scaled])
colors = {"malignant": "#d62728", "benign": "#1f77b4"}

plt.figure(figsize=(8, 8))

for diagnosis in np.unique(labels):
    idx = labels == diagnosis
    jitter = np.random.default_rng(RANDOM_SEED).normal(0, 0.08, size=(idx.sum(), 2))
    plt.scatter(
        winners[idx, 0] + jitter[:, 0],
        winners[idx, 1] + jitter[:, 1],
        label=diagnosis,
        c=colors[diagnosis],
        alpha=0.7,
        edgecolors="white",
        linewidths=0.4,
    )

plt.title("Breast Cancer Samples Mapped onto the SOM")
plt.xlabel("SOM x")
plt.ylabel("SOM y")
plt.xticks(range(MAP_SIZE[0]))
plt.yticks(range(MAP_SIZE[1]))
plt.grid(True, alpha=0.25)
plt.legend(title="Diagnosis")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
first_step, first_matrix = snapshots[0]
mesh = ax.pcolor(first_matrix, cmap="viridis", vmin=0, vmax=max(frame.max() for _, frame in snapshots))
colorbar = fig.colorbar(mesh, ax=ax, label="Mean neighbor distance")

def update(frame_index):
    step, matrix = snapshots[frame_index]
    ax.clear()
    mesh = ax.pcolor(matrix, cmap="viridis", vmin=0, vmax=max(frame.max() for _, frame in snapshots))
    ax.set_title(f"SOM U-Matrix Learning - iteration {step:,}")
    ax.set_xlabel("SOM x")
    ax.set_ylabel("SOM y")
    ax.set_xticks(range(MAP_SIZE[0] + 1))
    ax.set_yticks(range(MAP_SIZE[1] + 1))
    return (mesh,)

animation = FuncAnimation(fig, update, frames=len(snapshots), interval=120, blit=False)
animation.save(ANIMATION_PATH, writer=PillowWriter(fps=10))
plt.close(fig)

print(f"Saved animation to: {ANIMATION_PATH.resolve()}")

try:
    from IPython.display import Image, display
    display(Image(filename=str(ANIMATION_PATH)))
except ModuleNotFoundError:
    print("Open the GIF file above to view the animation.")